# Legislative Content Extraction (ADK) - Evaluation Smoke Test

Fetch the dataset from Langfuse and run the agent on the first item to verify the pipeline works end-to-end.

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

# Ensure cwd is the repo root so relative paths work
REPO_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd()
# Walk up until we find pyproject.toml
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

load_dotenv(verbose=True)
print(f"Working directory: {Path.cwd()}")

## 1. Fetch dataset from Langfuse

In [ ]:
from aieng.agent_evals.async_client_manager import AsyncClientManager

client_manager = AsyncClientManager.get_instance()
langfuse_client = client_manager.langfuse_client

DATASET_NAME = "legislative-content-extraction"
dataset = langfuse_client.get_dataset(DATASET_NAME)
print(f"Dataset: {dataset.name}")
print(f"Items:   {len(dataset.items)}")

## 2. Inspect the first item

In [ ]:
FILES_DIR = Path("implementations/legislative_content_extraction/files").resolve()

# Find the first item that has a PDF on disk
first_item = None
for item in dataset.items:
    record_id = item.input["record_id"]
    record_dir = FILES_DIR / record_id
    if record_dir.is_dir() and list(record_dir.glob("*.pdf")):
        first_item = item
        break

assert first_item is not None, "No dataset item has a local PDF"

print("Input:")
print(json.dumps(first_item.input, indent=2))
print("\nExpected output:")
eo = first_item.expected_output
for key in ["jurisdiction_code", "session_code", "chamber_code", "measure_type_code", "measure_number"]:
    print(f"  {key}: {eo.get(key)}")
print(f"  sponsors: {eo.get('sponsors')}")
print(f"  sections_affected: {len(eo.get('sections_affected', []))} entries")

## 3. Run the agent on the first item

In [ ]:
from aieng.agent_evals.legislative_content_extraction import (
    LegislativeContentExtractionAgent,
)

input_data = first_item.input
record_id = input_data["record_id"]
html_page_link = input_data.get("html_page_link", "")
prompt = input_data["prompt"]

pdf_path = str(list((FILES_DIR / record_id).glob("*.pdf"))[0])

print(f"Record:  {record_id}")
print(f"PDF:     {pdf_path}")
print(f"HTML:    {html_page_link or '(none)'}")

In [ ]:
agent = LegislativeContentExtractionAgent(
    files_dir=FILES_DIR,
)

response = await agent.answer_async(
    pdf_path=pdf_path,
    prompt=prompt,
    html_page_link=html_page_link,
)

print(f"Duration: {response.total_duration_ms}ms")
print(f"Tool calls: {len(response.tool_calls)}")
for tc in response.tool_calls:
    print(f"  - {tc['name']}({list(tc.get('args', {}).keys())})")

## 4. Compare output vs expected

In [ ]:
output = json.loads(response.text)
expected = first_item.expected_output

exact_fields = [
    "jurisdiction_code", "session_code", "chamber_code",
    "measure_type_code", "measure_number",
]

print(f"{'Field':<25} {'Expected':<25} {'Got':<25} {'Match'}")
print("-" * 80)
for field in exact_fields:
    exp_val = str(expected.get(field, ""))
    got_val = str(output.get(field, ""))
    match = exp_val.strip().lower() == got_val.strip().lower()
    symbol = "OK" if match else "MISMATCH"
    print(f"{field:<25} {exp_val:<25} {got_val:<25} {symbol}")

In [ ]:
print("Agent extracted metadata:")
print(json.dumps(output, indent=2))

In [ ]:
await client_manager.close()